<img src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/AIFS-M03-D03-ETL_business_case.png" />

# PeopleFirst ETL pipeline 🔄

You work as a junior data engineer at **PeopleFirst Consulting**. The CFO asked the HR team to track bonuses on top of base salaries. The HR director drops a new daily export from TalentCloud into a shared folder. Your job is to build the pipeline that moves this data from a raw cloud storage layer into a clean PostgreSQL database.

You already know the theory. Now you walk through the full pipeline on real AWS, from the first S3 click to the final SQL report.

<Note type="important" title="Just follow the steps!">

In this exercise, you will follow the steps to set up an S3 bucket, create a PostgreSQL database on AWS RDS, and run a Python ETL script that moves the data from S3 to PostgreSQL. You will not write business SQL queries in this notebook. You will only follow the steps to set up the infrastructure and run the pipeline. Just follow the steps.

No solutions are provided for this notebook. If you get stuck, ask for help to your classmates or to your TA.

The goal of this exercise is to get familiar with the full ETL flow on real AWS: storage in S3, a Python cleaning step, and a destination in RDS PostgreSQL. You will do this many times in your career, so it is important to understand the steps and the tools involved.

</Note>

## The pipeline you will build
For this exercise, you will build a simple ETL pipeline that moves employee data from a raw JSON file in S3 into a clean table in RDS PostgreSQL. The pipeline will run on your laptop using Python.

You need three tools:

| Tool                  | Role                              |
| --------------------- | --------------------------------- |
| Amazon S3             | Stores the raw JSON files         |
| Amazon RDS PostgreSQL | Stores the clean employee table   |
| Python on your laptop | Reads, cleans, and loads the data |

## The raw data

Download the new daily export: [raw_2026_03_17.zip](https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/raw_2026_03_17.zip)

Unzip it on your computer. You should see a folder organized like this:

```text
raw/
└── employees/
    └── 2026/
        └── 03/
            └── 17/
                ├── employee_001.json
                ├── employee_002.json
                └── ...
```

Each JSON file holds one employee record. Example:

```json
{
  "employeeId": "EMP-04217",
  "team": "Data & Analytics",
  "seniorityLevel": "senior",
  "baseSalary": "72500.00 EUR",
  "bonus": "5800.00 EUR",
  "status": "active",
  "contractType": "PERMANENT"
}
```

## Step 1: Create an S3 bucket

Go to the AWS Console. Open S3. Click `Create bucket`.

Use these settings:

| Setting                 | Value                                  |
| ----------------------- | -------------------------------------- |
| Bucket name             | `peoplefirst-hr-bonus`    |
| Region                  | `eu-west-3`                            |

Click `Create bucket`.

S3 bucket names must be unique across all AWS accounts. Add your initials and the year if the name is taken.

## Step 2: Upload the JSON files to S3

Open your bucket. Click `Upload`, then `Add folder`. Select the local `raw/` folder you unzipped. Confirm the upload.

After the upload, the files should appear in S3 under this path:

```text
raw/employees/2026/03/17/
```

Click on one of the JSON files to confirm it opens and shows a real employee record.

## Step 3: Create the RDS PostgreSQL database

Go to the AWS Console. Open RDS. Click `Create database`.

Use these settings:

| Setting                | Value                                      |
| ---------------------- | ------------------------------------------ |
| Creation method        | Standard create                            |
| Engine                 | PostgreSQL                                 |
| Template               | Free tier if available, otherwise Dev/Test |
| DB instance identifier | `peoplefirst-postgres-v2`                  |
| Master username        | `postgres`                                 |
| Master password        | Choose a password and save it              |
| DB instance class      | Smallest available option                  |
| Storage                | 20 GB                                      |
| Public access          | `Yes`                                      |
| VPC                    | Create new → name it `peoplefirst-sg`      |
| Database name          | `peoplefirst`                              |

Click `Create database`. Wait until the status shows `Available`. This takes a few minutes.

<Note type="important" title="Public access is required for this lab">

Public access is acceptable for this short lab. Never do this in production.

</Note>

## Step 4: Copy your RDS connection details

Open your RDS database page. Copy the endpoint. It looks like this:

```text
peoplefirst-postgres-v2.xxxxxxxxxxxx.eu-west-3.rds.amazonaws.com
```

Keep these five values nearby:

| Information | Example                                                              |
| ----------- | -------------------------------------------------------------------- |
| Host        | `peoplefirst-postgres-v2.xxxxxxxxxxxx.eu-west-3.rds.amazonaws.com`   |
| Port        | `5432`                                                               |
| Database    | `peoplefirst`                                                        |
| User        | `postgres`                                                           |
| Password    | Your RDS password                                                    |

## Step 5: Connect to PostgreSQL from Python

You will connect to the database directly from Python with `psycopg2`. This way, every step of the pipeline lives in your script.

Run this code with the five values from Step 4.

In [9]:
import os
import psycopg2
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

RDS_HOST = os.getenv("DB_HOST")
RDS_PORT = 5432
RDS_DBNAME = "peoplefirst"
RDS_USER = "postgres"
RDS_PASSWORD = getpass("RDS password: ")   # comme dans l'exercice : saisi, jamais écrit

pg_conn = psycopg2.connect(
    host=RDS_HOST,
    port=RDS_PORT,
    dbname=RDS_DBNAME,
    user=RDS_USER,
    password=RDS_PASSWORD,
    connect_timeout=10,          # ajout : échoue en 10 s au lieu d'attendre longtemps
)
pg_cursor = pg_conn.cursor()
print("Connected to RDS PostgreSQL.")

Connected to RDS PostgreSQL.


## Step 6: Create the employees_v2 table

You will send the `CREATE TABLE` query through the cursor you opened in Step 5.

Run this code:

In [2]:
# Drop the table if it already exists, so the script can run again cleanly
pg_cursor.execute("DROP TABLE IF EXISTS employees_v2;")

# Create the destination table
create_table_sql = """
CREATE TABLE employees_v2 (
    employee_id     VARCHAR(20) PRIMARY KEY,
    team            VARCHAR(100),
    seniority_level VARCHAR(20),
    base_salary     NUMERIC(10, 2),
    bonus           NUMERIC(10, 2),
    status          VARCHAR(20),
    contract_type   VARCHAR(20)
);
"""
pg_cursor.execute(create_table_sql)

# Save the change in the database
pg_conn.commit()

print("Table employees_v2 created on RDS.")

Table employees_v2 created on RDS.


## Step 7: Create the Python project folder

Open a terminal. Run these commands:

```bash
mkdir peoplefirst-etl-v2
cd peoplefirst-etl-v2
python -m venv .venv
```

Activate the virtual environment.

On macOS or Linux:

```bash
source .venv/bin/activate
```

On Windows PowerShell:

```powershell
.venv\Scripts\Activate.ps1
```

In your terminal, install the three Python libraries:

```bash
pip install boto3 psycopg2-binary python-dotenv
```

Put this notebook inside the `peoplefirst-etl-v2` folder. 

## Step 8: Create the .env file

<img src="https://ai-fullstack-assets.s3.eu-west-3.amazonaws.com/M03-Data_Infrastructure/AIFS-M03-D03-ETL_env_variables.png" />

Inside the project folder, create a file called `.env`. Paste this content:

```text
S3_BUCKET=peoplefirst-hr-bonus
S3_PREFIX=raw/employees/2026/03/17/

AWS_ACCESS_KEY_ID=...your AWS access key ID...
AWS_SECRET_ACCESS_KEY=...your AWS secret access key...
AWS_DEFAULT_REGION=eu-west-3

DB_HOST=peoplefirst-postgres-v2.xxxxxxxxxxxx.eu-west-3.rds.amazonaws.com
DB_PORT=5432
DB_NAME=peoplefirst
DB_USER=postgres
DB_PASSWORD=your-rds-password
```

Replace every value with your own bucket name, RDS endpoint, and password.

## Step 9: Create the etl.py file

Inside the project folder, create a file called `etl.py`. Paste this code:

```python
import json
import os
from decimal import Decimal

import boto3
import psycopg2
from dotenv import load_dotenv


load_dotenv()

S3_BUCKET = os.getenv("S3_BUCKET")
S3_PREFIX = os.getenv("S3_PREFIX")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

# The extract step is where you read the raw data from S3. You will use the `boto3` library to connect to S3, list the JSON files in the specified prefix, and read their contents into a list of Python dictionaries.
def extract_from_s3():
    s3 = boto3.client("s3")
    employees_raw = []

    response = s3.list_objects_v2(
        Bucket=S3_BUCKET,
        Prefix=S3_PREFIX,
    )

    items = response.get("Contents", [])
    for item in items:
        key = item["Key"]

        if not key.endswith(".json"):
            continue

        s3_object = s3.get_object(
            Bucket=S3_BUCKET,
            Key=key,
        )

        body_bytes = s3_object["Body"].read()
        body_text = body_bytes.decode("utf-8")
        employee = json.loads(body_text)

        employees_raw.append(employee)

    return employees_raw

# The transform step is where you clean the data and prepare it for loading into PostgreSQL.
# In this example, you will:
# - Keep only active employees
# - Remove interns
# - Convert base salary and bonus from text to decimal
def transform(employees_raw):
    employees_clean = []

    for employee in employees_raw:
        status = employee["status"]
        if status != "active":
            continue

        contract_type = employee["contractType"]
        if contract_type == "INTERN":
            continue

        salary_text = employee["baseSalary"]
        salary_parts = salary_text.split(" ")
        salary_amount_text = salary_parts[0]
        base_salary = Decimal(salary_amount_text)

        bonus_text = employee["bonus"]
        bonus_parts = bonus_text.split(" ")
        bonus_amount_text = bonus_parts[0]
        bonus = Decimal(bonus_amount_text)

        clean_row = {
            "employee_id": employee["employeeId"],
            "team": employee["team"],
            "seniority_level": employee["seniorityLevel"],
            "base_salary": base_salary,
            "bonus": bonus,
            "status": status,
            "contract_type": contract_type,
        }

        employees_clean.append(clean_row)

    return employees_clean

# The load step is where you insert the clean data into PostgreSQL. You will connect to the database, then run an `INSERT` query for each employee. If an employee with the same ID already exists, you will update their record instead.
def load_to_postgres(employees_clean):
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
    )

    cur = conn.cursor()

    sql = """
    INSERT INTO employees_v2 (
        employee_id,
        team,
        seniority_level,
        base_salary,
        bonus,
        status,
        contract_type
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (employee_id) DO UPDATE SET
        team = EXCLUDED.team,
        seniority_level = EXCLUDED.seniority_level,
        base_salary = EXCLUDED.base_salary,
        bonus = EXCLUDED.bonus,
        status = EXCLUDED.status,
        contract_type = EXCLUDED.contract_type;
    """

    for row in employees_clean:
        values = (
            row["employee_id"],
            row["team"],
            row["seniority_level"],
            row["base_salary"],
            row["bonus"],
            row["status"],
            row["contract_type"],
        )
        cur.execute(sql, values)

    conn.commit()
    cur.close()
    conn.close()

# The main function orchestrates the ETL flow. It calls the extract, transform, and load functions in sequence, and prints some logs along the way.
def main():
    print("Extracting data from S3...")
    employees_raw = extract_from_s3()
    raw_count = len(employees_raw)
    print(f"Raw employees found: {raw_count}")

    print("Transforming data...")
    employees_clean = transform(employees_raw)
    clean_count = len(employees_clean)
    print(f"Clean employees ready to load: {clean_count}")

    print("Loading data into PostgreSQL...")
    load_to_postgres(employees_clean)

    print("ETL done.")


if __name__ == "__main__":
    main()
```

## Step 10: Run the ETL script

In the terminal, with the virtual environment active, run:

```bash
python etl.py
```

You should see output similar to this:

```text
Extracting data from S3...
Raw employees found: 50
Transforming data...
Clean employees ready to load: 37
Loading data into PostgreSQL...
ETL done.
```

## Step 11: Check the data in PostgreSQL

Rerun step 5 to reconnect to your PostgreSQL database. You will run three quick checks through the cursor. Each check tells you one thing about the loaded data.

**Check 1: how many rows did you load?**

In [3]:
pg_cursor.execute("SELECT COUNT(*) FROM employees_v2;")
row_count = pg_cursor.fetchone()
print("Rows in employees_v2:", row_count[0])

Rows in employees_v2: 37


**Check 2: do the columns look right?**

In [4]:
pg_cursor.execute("SELECT * FROM employees_v2 LIMIT 5;")
sample_rows = pg_cursor.fetchall()

print("First five rows:")
for row in sample_rows:
    print(row)

First five rows:
('EMP-04201', 'Finance', 'junior', Decimal('35409.00'), Decimal('2518.00'), 'active', 'PERMANENT')
('EMP-04202', 'Finance', 'junior', Decimal('43935.00'), Decimal('1178.00'), 'active', 'PERMANENT')
('EMP-04203', 'Data & Analytics', 'mid', Decimal('53811.00'), Decimal('4569.00'), 'active', 'PERMANENT')
('EMP-04204', 'Finance', 'lead', Decimal('97223.00'), Decimal('11679.00'), 'active', 'CONTRACT')
('EMP-04205', 'Data & Analytics', 'mid', Decimal('61438.00'), Decimal('4231.00'), 'active', 'PERMANENT')


**Check 3: did the filters work?**

In [5]:
pg_cursor.execute("SELECT DISTINCT status, contract_type FROM employees_v2;")
status_combinations = pg_cursor.fetchall()

print("Status and contract_type combinations in the table:")
for combo in status_combinations:
    print(combo)

Status and contract_type combinations in the table:
('active', 'CONTRACT')
('active', 'PERMANENT')


## Step 12: Run the final report

You now send the report query through the cursor. The result is a list of rows, one per team, sorted from the highest paid team to the lowest.

In [6]:
report_sql = """
SELECT
    team,
    COUNT(*) AS headcount,
    ROUND(AVG(base_salary + bonus), 2) AS avg_total_compensation
FROM employees_v2
WHERE status = 'active'
GROUP BY team
ORDER BY avg_total_compensation DESC;
"""

pg_cursor.execute(report_sql)
report_rows = pg_cursor.fetchall()

print("CFO report: average total compensation by team")
print("-" * 60)
for row in report_rows:
    team, headcount, avg_total = row
    print(f"{team:<25} | headcount: {headcount:>3} | avg: {avg_total} EUR")

CFO report: average total compensation by team
------------------------------------------------------------
People Operations         | headcount:  10 | avg: 80076.90 EUR
Finance                   | headcount:   8 | avg: 75608.88 EUR
Engineering               | headcount:   5 | avg: 66902.40 EUR
Sales                     | headcount:   5 | avg: 61546.20 EUR
Marketing                 | headcount:   3 | avg: 53759.67 EUR
Data & Analytics          | headcount:   6 | avg: 53467.50 EUR


You should see one row per team. The first row is the team that pays the most on average. That is the table the CFO asked for.

Once you read the report, close the cursor and the connection:

In [14]:
pg_cursor.close()
pg_conn.close()
print("Connection closed.")

Connection closed.


## Step 13: Add a data quality gate (and a Dead Letter Queue)

The next morning, the HR director calls. TalentCloud pushed a corrupted export overnight. Your `transform` function filters the statuses and contract types it knows about, but it never checks the values themselves. A negative salary would sail straight into `employees_v2` and poison the CFO's averages.

You will add a **data quality gate**: a checkpoint between transform and load that validates every row against an explicit contract and rejects the rows that break it. You will build it with **Pandera**, a Python library that declares rules for a pandas DataFrame (types, ranges, allowed values) and validates the whole frame in one call. This gate is exactly what criterion C3.4 of the AIA certification grades, and you will rebuild one on your own pipeline in the M13 final project.

Install the two libraries in both places where your code runs. In the terminal, inside `peoplefirst-etl-v2` with the virtual environment active:

```bash
pip install pandera pandas
```

Then run the cell below to install them for this notebook too.

In [ ]:
%pip install -q pandera pandas

### Define the data contract

Write down what a valid clean row looks like. Each rule encodes one business fact: employee IDs follow the `EMP-` format, pay is never negative, and only known statuses and contract types exist.

Run this cell. Pandera may print a `FutureWarning` about its import path. Ignore it in this lab.

In [1]:
import pandas as pd
import pandera as pa

# The data contract: every clean row must respect these rules
employee_schema = pa.DataFrameSchema(
    {
        "employee_id": pa.Column(str, pa.Check.str_startswith("EMP-")),
        "team": pa.Column(str),
        "seniority_level": pa.Column(str),
        # coerce=True converts the Decimal values from the transform step
        # into floats before checking them
        "base_salary": pa.Column(float, pa.Check.ge(0), coerce=True),
        "bonus": pa.Column(float, pa.Check.ge(0), coerce=True),
        "status": pa.Column(str, pa.Check.isin(["active"])),
        "contract_type": pa.Column(str, pa.Check.isin(["PERMANENT", "CONTRACT"])),
    }
)

# Quick test: the sample record from "The raw data" section, in its clean form
sample_row = pd.DataFrame([
    {
        "employee_id": "EMP-04217",
        "team": "Data & Analytics",
        "seniority_level": "senior",
        "base_salary": 72500.00,
        "bonus": 5800.00,
        "status": "active",
        "contract_type": "PERMANENT",
    }
])

employee_schema.validate(sample_row, lazy=True)
print("Contract defined. The sample record passes.")

Contract defined. The sample record passes.


/Users/thierrylegros/00-JEDHA/Fullstack-AI/.venv/lib/python3.14/site-packages/pandera/_pandas_deprecated.py:144: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


### Inject a poisoned record

Do not wait for TalentCloud to break again. Break the data yourself, on purpose, to prove the gate works. The cell below uses `boto3` to upload `employee_999.json` into the same S3 prefix as Step 2. The record carries two problems: a negative salary and a contract type that does not exist.

In [2]:
import json
import os

import boto3
from dotenv import load_dotenv

# Reuse the same .env file as etl.py (Step 8)
load_dotenv()

S3_BUCKET = os.getenv("S3_BUCKET")
S3_PREFIX = os.getenv("S3_PREFIX")

# The poisoned record: negative salary, unknown contract type
poison_record = {
    "employeeId": "EMP-00999",
    "team": "Data & Analytics",
    "seniorityLevel": "senior",
    "baseSalary": "-15000.00 EUR",
    "bonus": "2000.00 EUR",
    "status": "active",
    "contractType": "GHOST",
}

s3 = boto3.client("s3")
s3.put_object(
    Bucket=S3_BUCKET,
    Key=S3_PREFIX + "employee_999.json",
    Body=json.dumps(poison_record).encode("utf-8"),
)

print(f"Poison uploaded to s3://{S3_BUCKET}/{S3_PREFIX}employee_999.json")

Poison uploaded to s3://peoplefirst-hr-bonus-tl/raw/employees/2026/03/17/employee_999.json


### Wire the gate into the pipeline

Open `etl.py`. Make four small changes.

**1. Add two imports** next to the existing ones at the top:

```python
import pandas as pd
import pandera as pa
```

**2. Add the contract** right after the environment variables. It is the same schema you defined above:

```python
# The data contract: every clean row must respect these rules
employee_schema = pa.DataFrameSchema(
    {
        "employee_id": pa.Column(str, pa.Check.str_startswith("EMP-")),
        "team": pa.Column(str),
        "seniority_level": pa.Column(str),
        "base_salary": pa.Column(float, pa.Check.ge(0), coerce=True),
        "bonus": pa.Column(float, pa.Check.ge(0), coerce=True),
        "status": pa.Column(str, pa.Check.isin(["active"])),
        "contract_type": pa.Column(str, pa.Check.isin(["PERMANENT", "CONTRACT"])),
    }
)
```

**3. Add the quality gate** between `transform` and `load_to_postgres`:

```python
# The quality gate sits between transform and load. Good rows continue to
# the load step. Bad rows land in the rejected/ folder and never reach the
# database.
def quality_gate(employees_clean):
    df = pd.DataFrame(employees_clean)

    try:
        validated_df = employee_schema.validate(df, lazy=True)
        return validated_df.to_dict("records")
    except pa.errors.SchemaErrors as err:
        # failure_cases lists every broken rule with the index of the guilty row
        bad_indexes = err.failure_cases["index"].dropna().astype(int).unique()

        os.makedirs("rejected", exist_ok=True)
        rejected_df = df.loc[bad_indexes]
        rejected_df.to_csv("rejected/employees_rejected.csv", index=False)
        print(f"Quality gate: {len(rejected_df)} row(s) quarantined in rejected/")

        clean_df = df.drop(index=bad_indexes)
        return clean_df.to_dict("records")
```

`lazy=True` tells Pandera to check every rule on every row and collect all the failures, instead of stopping at the first one. Pandera then raises one `SchemaErrors` exception carrying a `failure_cases` table: one line per broken rule, with the index of the row that broke it. The `except` block uses those indexes to split the batch: bad rows go to a local `rejected/` folder as a CSV, clean rows continue to the load step.

That `rejected/` folder is your homemade **Dead Letter Queue (DLQ)**: the place where a pipeline parks the records it cannot process, so a human can inspect them later instead of crashing the run or silently losing data. In industry, teams build the same gate at scale with **Great Expectations**, the reference data validation framework; Pandera is its lightweight cousin.

**4. Replace the `main` function** with this version, which routes the data through the gate:

```python
def main():
    print("Extracting data from S3...")
    employees_raw = extract_from_s3()
    print(f"Raw employees found: {len(employees_raw)}")

    print("Transforming data...")
    employees_clean = transform(employees_raw)
    print(f"Clean employees after transform: {len(employees_clean)}")

    print("Running the quality gate...")
    employees_validated = quality_gate(employees_clean)
    print(f"Validated employees ready to load: {len(employees_validated)}")

    print("Loading data into PostgreSQL...")
    load_to_postgres(employees_validated)

    print("ETL done.")
```

### Rerun the pipeline

In the terminal, run:

```bash
python etl.py
```

You should see output similar to this:

```text
Extracting data from S3...
Raw employees found: 51
Transforming data...
Clean employees after transform: 38
Running the quality gate...
Quality gate: 1 row(s) quarantined in rejected/
Validated employees ready to load: 37
Loading data into PostgreSQL...
ETL done.
```

The run did not crash. The poison went through extract and transform, then the gate caught it. Now look inside the quarantine:

In [3]:
import pandas as pd

rejected_df = pd.read_csv("rejected/employees_rejected.csv")
print("Rows in quarantine:", len(rejected_df))
rejected_df

Rows in quarantine: 1


,employee_id,team,seniority_level,base_salary,bonus,status,contract_type
0,EMP-00999,Data & Analytics,senior,-15000.0,2000.0,active,GHOST


You should see exactly one row: `EMP-00999`, with its negative salary and its `GHOST` contract type. Nothing was lost: the record sits in the DLQ, ready for a human to inspect and send back to TalentCloud.

Last check: confirm the poison never reached the database. Rerun the cell in Step 5 to reconnect, then run the checks below. You should get 37 rows and 0 rows breaking the contract.

In [6]:
pg_cursor.execute("SELECT COUNT(*) FROM employees_v2;")
print("Rows in employees_v2:", pg_cursor.fetchone()[0])

pg_cursor.execute("""
    SELECT COUNT(*)
    FROM employees_v2
    WHERE base_salary < 0
       OR contract_type NOT IN ('PERMANENT', 'CONTRACT');
""")
print("Rows breaking the contract:", pg_cursor.fetchone()[0])

Rows in employees_v2: 37
Rows breaking the contract: 0


## Step 14: Anonymize the PII and trace the lineage

The CFO loved the report. Now an external benchmarking firm asks for the full compensation table, to compare PeopleFirst against the market. Look at the columns of `employees_v2` before you send anything. `team`, `seniority_level`, and the pay columns describe groups of people. `employee_id` points at one specific human. That makes it **PII** (Personally Identifiable Information: any value that can be traced back to one specific person). Salary data attached to a person's identifier is exactly what **GDPR** (the European data protection law) forbids you to share carelessly.

You will replace each real ID with a **salted hash**: a scrambled fingerprint produced by a one-way function (SHA-256 here), mixed with a secret string called the **salt**. The same ID always produces the same fingerprint, so the data stays usable for analysis. But without the salt, nobody can walk back from the fingerprint to the person. Privacy protection inside the pipeline is criterion C3.5 of the AIA certification, the lineage trace at the end of this step is criterion C3.7, and you will redo both on your own pipeline in the M13 final project.

<Note type="important" title="Add the salt to your .env file">

Open the `.env` file from Step 8 and add one line at the end:

```text
HASH_SALT=invent-a-long-random-string-and-keep-it-secret
```

The salt is a secret, exactly like your database password. It lives in `.env` only: never in the code, never in the notebook, never on GitHub.

</Note>

Then run the cell below.

In [8]:
import hashlib
import os

from dotenv import load_dotenv

# override=True forces a fresh read of .env, so the new HASH_SALT line is picked up
load_dotenv(override=True)

HASH_SALT = os.getenv("HASH_SALT")
if HASH_SALT is None:
    raise ValueError("HASH_SALT is missing. Add it to your .env file first.")


def pseudonymize(employee_id):
    # The salt is glued to the ID before hashing. Without the salt, an attacker
    # who knows the EMP-xxxxx format cannot rebuild the fingerprints one by one.
    salted_text = HASH_SALT + employee_id
    return hashlib.sha256(salted_text.encode("utf-8")).hexdigest()


print("Real ID:   EMP-04217")
print("Pseudonym:", pseudonymize("EMP-04217"))

Real ID:   EMP-04217
Pseudonym: c21929f887d1698390a81945012802b5dce0cbeb5a7aae7d5f2f29b88082b5e1


Run the cell twice. The fingerprint does not change: same input, same salt, same output. That stability is what keeps the data usable for joins and analysis.

Now produce the export for the benchmarking firm. The IDs are replaced before the file is even written to disk. Protecting data at the earliest possible point, instead of patching it at the last minute, is called **Privacy by Design**. Your connection from Step 5 should still be open from the Step 13 checks; rerun Step 5 if it is not.

In [10]:
import pandas as pd

pg_cursor.execute("""
    SELECT employee_id, team, seniority_level, base_salary, bonus
    FROM employees_v2;
""")
rows = pg_cursor.fetchall()

shareable_df = pd.DataFrame(
    rows,
    columns=["employee_id", "team", "seniority_level", "base_salary", "bonus"],
)

# Replace every real ID with its salted fingerprint
shareable_df["employee_id"] = shareable_df["employee_id"].apply(pseudonymize)

shareable_df.to_csv("employees_shareable.csv", index=False)
print("Shareable export written to employees_shareable.csv")
shareable_df.head()

Shareable export written to employees_shareable.csv


,employee_id,team,seniority_level,base_salary,bonus
0,a378e53c556ec32e20a971c6df5d92de1cdfe02b60604a...,Finance,junior,35409.00,2518.00
1,9e41476a2cc07d08df946afd48a519f4dd48a7ecb8f3de...,Finance,junior,43935.00,1178.00
2,5c8acb380cd70d37f63c86634bc7bc20866fa477ad8207...,Data & Analytics,mid,53811.00,4569.00
3,a3c2228028e7755f22b5c7e8577e51fd0d009f68a4f0f2...,Finance,lead,97223.00,11679.00
4,7a4315a9474f72d706d8e3fe694a57baf6103e955b24dc...,Data & Analytics,mid,61438.00,4231.00


<Note type="definition" title="Masking, pseudonymization, dropping">

**Masking** hides part of a value and keeps the rest readable, like showing `EMP-***17` on a support screen. **Pseudonymization** replaces the value with a consistent stand-in, like your salted hash: analysis still works, and whoever holds the salt can re-link each stand-in to the person. **Dropping** deletes the column entirely: no salt, no way back, and it is the only true anonymization of the three. Under GDPR, a salted hash is still personal data, precisely because the salt holder can rebuild the link. That is why the salt stays secret in `.env` and never travels with the data.

</Note>

### Trace the lineage

Last deliverable. The benchmarking firm's auditor asks a simple question: "where does each column in this file come from?" You answer with a **lineage table**: one row per final column, tracing it from the S3 source field to the transformations that produced it.

The first two rows are done. Complete the other five by editing this cell (double-click it):

| Final column | Source field in `raw/employees/2026/03/17/*.json` | Transformations applied |
| --- | --- | --- |
| `employee_id` | `employeeId` | Renamed to snake_case; checked against the `EMP-` format by the quality gate; pseudonymized with a salted SHA-256 hash in the shareable export |
| `team` | `team` | Renamed only |
| `seniority_level` | ... | ... |
| `base_salary` | ... | ... |
| `bonus` | ... | ... |
| `status` | ... | ... |
| `contract_type` | ... | ... |

This trace is **data lineage**: the record of where each piece of data comes from and what happened to it along the way. In industry, tools like **OpenLineage** generate it automatically.

You are done with the database. Rerun the last cell of Step 12 to close the connection, then move on to Step 15 to delete your AWS resources.

## Step 15: Delete the AWS resources

RDS keeps generating costs even when you are not using it. **Delete everything you created**.

Go to RDS. Select your database. Click `Actions`, then `Delete`. Skip the final snapshot. Confirm.

Go to S3. Open your bucket. Empty the bucket first, then delete it.

<Note type="important" title="RDS costs money">

If you forget to delete the database, your AWS account keeps getting charged.

Always delete RDS resources after a learning lab.

</Note>